# Phase 6 -- Model Selection & Saving

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score
from imblearn.over_sampling import SMOTE
import warnings

warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('credit_card_fraud_dataset.csv')
print(f"Dataset shape: {df.shape}")

### 6.1 -- Run Full Preprocessing & SMOTE Pipeline

In [ ]:
# Encoding
day_mapping = {'Mon': 0, 'Tue': 1, 'Wed': 2, 'Thu': 3, 'Fri': 4, 'Sat': 5, 'Sun': 6}
df['day_of_week'] = df['day_of_week'].map(day_mapping)

le_merchant = LabelEncoder()
df['merchant_category'] = le_merchant.fit_transform(df['merchant_category'])

# Separate features and target
df.drop('transaction_id', axis=1, inplace=True)
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling (fit ONLY on training data)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Balance training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("Preprocessing pipeline successfully executed.")

### 6.2 -- Train Models & Select Best Classifier

We compare the trained models focusing on Recall (catching as many fraudulent transactions as possible) and F1-Score.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

best_recall = -1
best_model_name = None
best_model = None
model_metrics = {}

for name, model in models.items():
    # Train
    model.fit(X_train_resampled, y_train_resampled)
    
    # Evaluate
    y_pred = model.predict(X_test_scaled)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    model_metrics[name] = {'Recall': recall, 'F1-Score': f1}
    
    print(f"{name}: Recall = {recall:.4f}, F1-Score = {f1:.4f}")
    
    # Prioritize Recall; use F1-Score as secondary decision maker
    if recall > best_recall:
        best_recall = recall
        best_model_name = name
        best_model = model
    elif recall == best_recall:
        # Tie-breaker using F1-score
        if f1 > model_metrics[best_model_name]['F1-Score']:
            best_model_name = name
            best_model = model

print(f"\nSelected Model: {best_model_name} (Recall: {model_metrics[best_model_name]['Recall']:.4f}, F1: {model_metrics[best_model_name]['F1-Score']:.4f})")

### 6.3 -- Save Best Model & Preprocessing Artifacts

We serialize the chosen model, the fitted scaler, and the encoders dictionary to the `models/` directory.

In [ ]:
os.makedirs('models', exist_ok=True)

# Save Best Model
joblib.dump(best_model, 'models/best_model.pkl')
print("Saved: models/best_model.pkl")

# Save Scaler
joblib.dump(scaler, 'models/scaler.pkl')
print("Saved: models/scaler.pkl")

# Save Encoders and day mapping
encoders = {
    'merchant_category_encoder': le_merchant,
    'day_of_week_mapping': day_mapping
}
joblib.dump(encoders, 'models/label_encoder.pkl')
print("Saved: models/label_encoder.pkl")

---
### Phase 6 Summary

- Compared Logistic Regression, Decision Tree, and Random Forest models.
- Selected the best model by prioritizing high Recall (to minimize false negatives in fraud cases) and F1-Score.
- Serialized the trained classifier (`best_model.pkl`), the feature scaler (`scaler.pkl`), and categorical mappings (`label_encoder.pkl`) to the `models/` directory.

Ready for Phase 7 (Streamlit Dashboard Development).

---